In [ ]:
import matplotlib.pyplot as plt
import re
import pandas as pd

In [ ]:
# To have logging in notebook

import logging

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
    datefmt="%H:%M:%S",
    force=True  # 👈 écrase toute config précédente
)

logger = logging.getLogger(__name__)


# Accessing Storage Files: Bronze → Silver
Explore how to access and manipulate job files stored in the bronze and silver layers using `get_storage_from_env()` and the project environment configuration.

In [ ]:
# Map du python path dans le docker
#import sys, os
#sys.path.insert(0, os.path.abspath('../..'))  # remonte à la racine du projet
#from src.config.env import load_project_env

## Load Project Environment
Import and load the project environment configuration to initialize environment variables.

In [ ]:
# Deactivate warning
import warnings
warnings.filterwarnings('ignore')

# Load project environment
from src.config.env import load_project_env
load_project_env()  # Safe to call multiple times (idempotent)
print("✅ Project environment loaded successfully")

from src.config.env import load_project_env
load_project_env()  # safe à rappeler (idempotent)


## Storage connections to silver / merged

In [ ]:
from src.storage.storage import get_storage_from_env
import src.utils.merge_dataset_utils as merge_utils
storage_wttj = get_storage_from_env("silver", "merged")


## Load wttj parquet file with helper

In [ ]:
df = merge_utils.read_wttj_parquet_file_to_df(storage_wttj,"merged_ft_dt=2026-03-07_wttj_dt=2026-03-07.parquet")

In [ ]:
df.info()

## Contracts

In [ ]:
# contract_counts = df['contract_type'].value_counts()
contract_counts_by_source = df.groupby('source')['contract_type'].value_counts()

### Avant normalisation

In [ ]:
display(df_normalize_contract_counts_by_source.head(30))
#print(f" Libellé unique = {len(df_ft_contract_counts_by_source)}")


### Après normalisation 

In [ ]:
def normalize_contracts(df, patterns):
    """
    Normalise les types de contrat :
    - Extrait le type principal (CDI, CDD, Intérim, etc.)
    - Extrait le détail (durée, précision)
    - Stocke dans contract_normalized et contract_detail
    """
    def extract_type(value):
        if pd.isna(value):
            return 'Inconnu'
        for pattern, label in patterns:
            if re.search(pattern, str(value)):
                return label
        return str(value)  # garder la valeur originale si pas de match

    def extract_detail(value):
        if pd.isna(value):
            return None
        # Supprimer le pattern trouvé et retourner le reste
        remaining = str(value)
        for pattern, label in patterns:
            remaining = re.sub(pattern, '', remaining).strip()
        # Nettoyer les séparateurs résiduels (-, :, espaces)
        remaining = re.sub(r'^[\s\-:]+|[\s\-:]+$', '', remaining)
        return remaining if remaining else None
    
    df = df.copy()
    df['contract_normalized'] = df['contract_type'].apply(extract_type)
    df['contract_detail']     = df['contract_type'].apply(extract_detail)
    
    return df


In [ ]:
def plot_contract_by_source(df, source, contract_column):
    # Filtrer par source
    df_source = df[df['source'] == source]
    
    # Calcul en pourcentage
    contract_pct = df_source[contract_column].value_counts(normalize=True).mul(100).round(1)
    
    # Graphique
    fig, ax = plt.subplots(figsize=(10, 5))
    contract_pct.plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
    
    # Afficher les % sur les barres
    for i, v in enumerate(contract_pct):
        ax.text(i, v + 0.5, f'{v}%', ha='center', fontweight='bold')
    
    ax.set_title(f"Répartition des types de contrat — {source}", fontsize=13, fontweight='bold')
    ax.set_xlabel("Type de contrat")
    ax.set_ylabel("Pourcentage (%)")
    ax.set_ylim(0, contract_pct.max() + 10)
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()


In [ ]:
# Mapping des patterns => type normalisé
patterns = [
    (r'(?i)cdi',                'CDI'),
    (r'(?i)contrat à durée indéterminée',                'CDD'),
    (r'(?i)cdd',                'CDD'),
    (r'(?i)contrat à durée déterminée',                'CDD'),
    (r'(?i)profession\s+lib',   'Profession libérale'),
    (r'(?i)intér?im',           'Intérim'),
    (r'(?i)saisonnier',           'Saisonnier'),
    (r'(?i)profession commerciale',     'Profession commerciale'),
    (r'(?i)franchise',     'Franchise')    
]

# Apply
df_normalize = normalize_contracts(df, patterns)

df_normalize_contract_counts_by_source = df_normalize.groupby('source')['contract_normalized'].value_counts()
print(f"Modalités de contrats FT = {len(df_normalize_contract_counts_by_source.loc['FT'])}")
print(f"Modalités de contrats WTTJ = {len(df_normalize_contract_counts_by_source.loc['WTTJ'])}")

display(df_normalize_contract_counts_by_source.head(30))

plot_contract_by_source(df_normalize, 'FT', 'contract_normalized')
plot_contract_by_source(df_normalize, 'WTTJ', 'contract_normalized')


### Expérience

In [ ]:
experience_level_counts_by_source = df.groupby('source')['experience_level'].value_counts()
experience_description_counts_by_source = df.groupby('source')['experience_description'].value_counts()

### Avant normalisation

**FT**
- experienceExige => D : débutant accepté, E : l’expérience est exigée, S : l’expérience est souhaitée
- experienceLibelle => Libellé de l’expérience ex : Débutant accepté / 1 ans ... 
- experienceCommentaire => Commentaire sur l’expérience. Ex: Expérience dans la vente souhaitée

On a pas la bonne correspondance de colonne.
Il faut prendre `experience_level` pour `wttj` et `experience_description` pour FT


In [ ]:
display(experience_level_counts_by_source.head(30))


In [ ]:
display(experience_description_counts_by_source.head(30))

In [ ]:
# Définition des tranches d'expérience avec leurs indices
EXPERIENCE_LEVELS = [
    (0, 'Débutant',    [r'(?i)débutant', r'(?i)0 an', r'(?i)sans expérience']),
    (1, '0-1 an',      [r'(?i)^1 an', r'(?i)^6 mois', r'(?i)^1 mois', r'(?i)^3 mois', r'(?i)less_than_6_months', r'(?i)6_months_to_1_year']),
    (2, '1-2 ans',     [r'(?i)^2 an', r'(?i)1_to_2_years']),
    (3, '2-3 ans',     [r'(?i)^3 an', r'(?i)^24 mois', r'(?i)2_to_3_years' ]),
    (4, '3-5 ans',     [r'(?i)^4 an', r'(?i)^5 an', r'(?i)4_to_5_years', r'(?i)3_to_4_years']),
    (5, '5-10 ans',    [r'(?i)^6 an', r'(?i)^7 an', r'(?i)^8 an', r'(?i)^9 an', r'(?i)^10 an', r'(?i)5_to_7_years', r'(?i)7_to_10_years']),
    (6, '10+ ans',     [r'(?i)^1[1-9] an', r'(?i)^[2-9][0-9] an', r'(?i)10_to_15_years', r'(?i)more_than_15_years']),
    (-1, 'Non précisé', [r'(?i)expérience exigée', r'(?i)expérience souhaitée']),
   
]

def normalize_experience(df, experience_col):
    """
    Normalise les niveaux d'expérience :
    - experience_normalized : label lisible (ex: '0-1 an')
    - experience_index      : indice numérique (ex: 1) pour trier/comparer
    - experience_detail     : valeur originale nettoyée
    """

    def extract_experience(value):
        if pd.isna(value):
            #To DEBUG
            return -1, 'NAN', None
            #return -1, 'Non précisé', None
        
        str_value = str(value).strip()
        
        for index, label, patterns in EXPERIENCE_LEVELS:
            for pattern in patterns:
                if re.search(pattern, str_value):
                    # Détail = valeur originale
                    return index, label, str_value
        
        #return -1, 'Non précisé', str_value
        # To debug
        return -1, value, str_value
    

    results = df[experience_col].apply(extract_experience)
    
    df = df.copy()
    df['experience_index']      = results.apply(lambda x: x[0])
    df['experience_normalized'] = results.apply(lambda x: x[1])
    df['experience_detail']     = results.apply(lambda x: x[2])
    
    return df


In [ ]:
def get_experience_col(row):
    if row['source'] == 'FT':
        return row['experience_description']
    elif row['source'] == 'WTTJ':
        return row['experience_level']
    else:
        return None 

df_normalize['experience_source_composite'] = df.apply(get_experience_col, axis=1)

# Utilisation
df_normalize = normalize_experience(df_normalize,'experience_source_composite')

# Vérification
#df_normalize[['experience_source_composite', 'experience_index', 'experience_normalized', 'experience_detail']].head(20)

display(df_normalize.groupby('source')['experience_normalized'].value_counts())

## ROME

In [ ]:
rome_count = df['rome_code'].value_counts()
print(rome_count)

# Status

In [ ]:
status_counts_by_source = df.groupby('source')['status'].value_counts()

print(status_counts_by_source)
#df.info()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# ─────────────────────────────────────────────
# 1. SUIVI STATUT PAR SOURCE
# ─────────────────────────────────────────────
def statut_par_source(df: pd.DataFrame) -> pd.DataFrame:
    """Tableau croisé : volume par source × statut."""
    return (
        df.groupby(["source", "status"])
        .size()
        .unstack(fill_value=0)
        .assign(TOTAL=lambda x: x.sum(axis=1))
        .sort_values("TOTAL", ascending=False)
    )

def unpublished_par_source(df: pd.DataFrame) -> pd.DataFrame:
    """Volume des offres unpublished par source."""
    return (
        df[df["status"].str.lower() == "unpublished"]
        .groupby("source")
        .size()
        .reset_index(name="nb_unpublished")
        .sort_values("nb_unpublished", ascending=False)
    )


# ─────────────────────────────────────────────
# 2. HISTOGRAMME DURÉE ACTIVE PAR SOURCE
# ─────────────────────────────────────────────

def histogramme_duree_active(
    df: pd.DataFrame,
    source: str,
    date_debut_col: str = "published_at",
    date_fin_col: str = "unpublished_at",
    bins: int = 30,
    max_days: int = 365,
) -> plt.Figure:
    """
    Histogramme de la durée active des annonces pour une source donnée.
    Si date_fin_col est vide, utilise updated_at comme proxy.
    """
    subset = df[df["source"] == source].copy()

    # Choisir la colonne de fin
    if subset[date_fin_col].notna().sum() > 0:
        fin = subset[date_fin_col]
        fin_label = date_fin_col
    else:
        fin = subset["updated_at"]
        fin_label = "updated_at (proxy)"

    subset["duree_jours"] = (fin - subset[date_debut_col]).dt.days
    duree = subset["duree_jours"].dropna()
    duree = duree[(duree >= 0) & (duree <= max_days)]

    if duree.empty:
        print(f"⚠️  Aucune donnée de durée pour : {source}")
        return None

    mediane = duree.median()
    moyenne  = duree.mean()
    n        = len(duree)

    PALETTE = {
        "bar": "#4C72B0", "median": "#DD8452",
        "mean": "#55A868", "bg": "#F8F9FA", "grid": "#E0E0E0",
    }

    fig, ax = plt.subplots(figsize=(10, 5))
    fig.patch.set_facecolor(PALETTE["bg"])
    ax.set_facecolor(PALETTE["bg"])

    ax.hist(duree, bins=bins, color=PALETTE["bar"],
            edgecolor="white", linewidth=0.6, alpha=0.9)

    ax.axvline(mediane, color=PALETTE["median"], linewidth=1.8,
               linestyle="--", label=f"Médiane : {mediane:.0f}j")
    ax.axvline(moyenne, color=PALETTE["mean"], linewidth=1.8,
               linestyle=":",  label=f"Moyenne : {moyenne:.0f}j")

    ax.set_title(
        f"Durée active des annonces — {source}\n"
        f"(n={n:,} | fin = {fin_label})",
        fontsize=13, fontweight="bold", pad=14,
    )
    ax.set_xlabel("Durée active (jours)", fontsize=11)
    ax.set_ylabel("Nombre d'annonces", fontsize=11)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
    ax.grid(axis="y", color=PALETTE["grid"], linewidth=0.8, zorder=0)
    ax.set_axisbelow(True)
    for spine in ["top", "right"]:
        ax.spines[spine].set_visible(False)

    ax.legend(frameon=False, fontsize=10)
    fig.tight_layout()
    return fig


def histogrammes_toutes_sources(df: pd.DataFrame, sources: list = None, **kwargs):
    """Lance histogramme_duree_active pour chaque source."""
    for src in (sources or df["source"].unique()):
        histogramme_duree_active(df, source=src, **kwargs)
        plt.show()



In [ ]:

# ─────────────────────────────────────────────────────────────────────────────
# Stats
# ─────────────────────────────────────────────────────────────────────────────
#
print(50*"─") 
print(' Statut × source')
print(50*"─") 
print(statut_par_source(df))

print(50*"─") 
print(' Volumes unpublished')
print(50*"─") 
print(unpublished_par_source(df))

# # Histogramme une source
#histogramme_duree_active(df, source="wttj")
#
# # Toutes les sources
#histogrammes_toutes_sources(df, bins=40, max_days=180)

## Statistics from helper

In [ ]:
if df is not None and not df.empty:
    merge_utils.print_statistics(df)
else    :
    logger.warning("⚠️ Aucune donnée chargée pour les statistiques FT/WTTJ fusionnées")